In [52]:
# import libraries
from pathlib import Path
import pandas as pd

# setup path to csv file
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path_raw_data = PROJECT_ROOT / "data" / "raw" / "PFAS Sample Sites - Surface Water and Fish Tissue.csv"

# read csv
df_raw = pd.read_csv(path_raw_data)

In [53]:
# first unpivot

# set columns to keep with unpivot
columns_ID = ['OBJECTID', 'COMMENTS']

# set columns to unpivot on
columns_unpivot = ['PFOS_MEASURE', 'PFOA_MEASURE']

# unpivot df
df_unpivot = df_raw.melt(
    id_vars=columns_ID,
    value_vars=columns_unpivot
)

df_unpivot.rename(columns={'variable': 'analyte'}, inplace=True)

# sort by objectid
df_unpivot.sort_values(by='OBJECTID', inplace=True)

In [54]:
# first split value columns

# when value column contains multiple values, split into one column per value
df_columns_split = df_unpivot['value'].str.split('/', expand=True)

# join split columns df with unpivoted df
df_unpivot = pd.concat([df_unpivot, df_columns_split], axis=1)

# remove original value column
df_unpivot.drop(columns='value', inplace=True)


In [55]:
# second unpivot

# set columns to keep with unpivot
columns_ID2 = columns_ID + ['analyte']

# unpivot again based of newly split value columns
df_unpivot = df_unpivot.melt(
    id_vars=columns_ID2,
    value_vars=[0, 1, 2, 3]
)

# remove variable column
df_unpivot.drop(columns='variable', inplace=True)

# remove rows with NaN in value column
df_unpivot.dropna(subset=['value'], inplace=True)

In [56]:
# second split columns

# split value column to separate month-day and result values
df_unpivot[['month-day', 'result']] = df_unpivot['value'].str.split(':', expand=True)

# remove value column
df_unpivot.drop(columns=['value'], inplace=True)

# df_unpivot

# df_unpivot[df_unpivot['OBJECTID']==2183]